# Week 21, Day 3: Numerical solvers: Euler, stability and Runge-Kutta

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w021/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Measure the order of each method

Write `euler_step(f, x, dt)`, `rk4_step(f, x, dt)` and `solve(f, x0, T, dt, step)` (return the state at time T). For a damped oscillator x'' = −x − 0.2x′ written as a two-variable system, compute the error at T = 10 against a very fine RK4 reference for dt = 0.05, 0.025, 0.0125 and 0.00625, and fit the slopes of log error against log dt.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def euler_step(f, x, dt):
    return ...

def rk4_step(f, x, dt):
    k1 = f(x)
    k2 = f(x + dt / 2 * k1)
    k3 = ...
    k4 = ...
    return ...

def solve(f, x0, T, dt, step):
    x = np.array(x0, dtype=float)
    for _ in range(int(round(T / dt))):
        x = step(f, x, dt)
    return x

def osc(x): return np.array([x[1], -x[0] - 0.2 * x[1]])
ref = solve(osc, [1.0, 0.0], 10.0, 0.001, rk4_step)
dts = np.array([0.05, 0.025, 0.0125, 0.00625])
err = {name: np.array([np.abs(solve(osc, [1.0, 0.0], 10.0, d, st) - ref).max() for d in dts]) for name, st in [("euler", euler_step), ("rk4", rk4_step)]}
order = {name: np.polyfit(np.log(dts), np.log(e), 1)[0] for name, e in err.items()}
print({k: v.round(8) for k, v in err.items()}); print("fitted orders:", {k: round(v, 2) for k, v in order.items()})
for name, e in err.items(): plt.loglog(dts, e, "o-", label=name)
plt.xlabel("dt"); plt.ylabel("error at T = 10"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_steps():
    "one RK4 step on dx/dt = -x from x = 1 with dt = 0.1 matches the known value"
    assert abs(rk4_step(lambda x: -x, np.array([1.0]), 0.1)[0] - 0.9048375) < 1e-7

def test_slopes():
    "the fitted orders are close to 1 for Euler and 4 for RK4"
    assert abs(order["euler"] - 1) < 0.15 and abs(order["rk4"] - 4) < 0.3

test_steps()
test_slopes()
print("All checks passed")

### Exercise: Explosion and its cure

With τ = 10 ms, V(0) = 1 and Δt = 25 ms, run 20 steps of forward Euler (`fwd`) and of backward Euler (`bwd`), each array including the starting value.

In [ ]:
tau, dt_big = 10.0, 25.0
fwd, bwd = [1.0], [1.0]
for _ in range(20):
    fwd.append(...)
    bwd.append(...)
fwd, bwd = np.array(fwd), np.array(bwd)
print("forward Euler:", fwd[:6].round(2), "...", f"{fwd[-1]:.3g}"); print("backward Euler:", bwd[:6].round(4))

In [ ]:
import numpy as np
def test_explodes():
    "forward Euler with dt = 25 ms explodes (|V| above 1000 after 20 steps)"
    assert abs(fwd[-1]) > 1000

def test_backward():
    "backward Euler stays positive and decays monotonically"
    assert np.all(bwd > 0) and np.all(np.diff(bwd) < 0)

test_explodes()
test_backward()
print("All checks passed")

### Exercise: A fast gate and a slow membrane

A gate m relaxes towards m∞(V) = 1/(1 + exp(−(V + 40)/5)) with τm = 0.1 ms, and the voltage obeys τ dV/dt = −(V + 70) + 30 m with τ = 20 ms. Starting from V = −70, m = 0, simulate 100 ms with forward Euler at dt = 0.5 ms (`m_fwd`, `V_fwd`) and with a semi-implicit scheme that updates m by backward Euler (m_new = (m + dt/τm · m∞(V)) / (1 + dt/τm)) and V by backward Euler (`V_bwd`). Compare with forward Euler at dt = 0.01 ms (`V_ref`).

In [ ]:
tau_m, tau_V = 0.1, 20.0
def m_inf(V): return 1 / (1 + np.exp(-(V + 40) / 5))

def run(dt, implicit):
    n = int(100 / dt)
    V, m = -70.0, 0.0
    Vs, ms = np.empty(n), np.empty(n)
    for i in range(n):
        if implicit:
            m = ...
            V = ...                                  # backward Euler for V with m fixed: (V + dt/tau_V (-70 + 30 m)) / (1 + dt/tau_V)
        else:
            m, V = m + dt / tau_m * (m_inf(V) - m), V + dt / tau_V * (-(V + 70) + 30 * m)
        Vs[i], ms[i] = V, m
        if abs(m) > 1e6: return Vs[:i + 1], ms[:i + 1]
    return Vs, ms

V_fwd, m_fwd = run(0.5, False)
V_bwd, m_bwd = run(0.5, True)
V_ref, m_ref = run(0.01, False)
print(f"forward Euler at 0.5 ms: largest |m| {np.abs(m_fwd).max():.3g};  semi-implicit final V {V_bwd[-1]:.3f} mV;  reference {V_ref[-1]:.3f} mV")

In [ ]:
import numpy as np
def test_forward_fails():
    "forward Euler with dt = 0.5 ms is unstable (the gate variable exceeds 10 in size)"
    assert np.abs(m_fwd).max() > 10

def test_backward_ok():
    "backward Euler with dt = 0.5 ms tracks the fine reference voltage to within 0.2 mV at the end"
    assert abs(V_bwd[-1] - V_ref[-1]) < 0.2

test_forward_fails()
test_backward_ok()
print("All checks passed")